<a href="https://colab.research.google.com/github/krishnakeshab-banik/credit_risk_modeling/blob/main/Credit_Risk_Scoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
sns.set_style("whitegrid")

In [ ]:
df=pd.read_csv("/content/german_credit_data.csv")

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df["Risk"].value_counts()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df["Job"].unique()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.dropna().reset_index(drop=True)

In [ ]:
df.drop(columns= 'Unnamed: 0', inplace= True)
#By using inplace=True, you instruct Pandas to: Update the existing object: The changes are applied to the variable itself in memory.

In [ ]:
df.head()

In [ ]:
df[["Age","Credit amount","Duration"]].hist(bins=20, edgecolor="black")
#bins means how many groups the data is segregated in
plt.suptitle("Distribution of Numerical Features", fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize= ( 10,5))
for i, col in enumerate (["Age","Credit amount","Duration"]):
  plt.subplot(1,3,i+1)
  sns.boxplot(y=df[col], color="skyblue")
  plt.title(col)
plt.tight_layout()
plt.show()

In [ ]:
df.query("Duration >= 60")

In [ ]:
categorical_cols=["Sex","Job","Housing","Saving accounts","Checking account","Purpose"]

In [ ]:
plt.figure(figsize= (10,10))
for i, col in enumerate(categorical_cols):
  plt.subplot(3,3,i+1)
  sns.countplot(data=df, x=col, palette="Set2", order= df[col].value_counts().index)
  plt.title(f"Distribution of {col}")
  plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
corr= df[["Age","Job","Credit amount","Duration"]].corr()

In [ ]:
corr

In [ ]:
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.show()

In [ ]:
df.groupby("Job")["Credit amount"].mean()

In [ ]:
df.groupby("Sex")["Credit amount"].mean()

In [ ]:
pd.pivot_table(df,values="Credit amount", index= "Housing", columns= "Purpose")

In [ ]:
sns.scatterplot(data=df,x="Age", y="Credit amount", hue="Sex", size="Duration", alpha= 0.7, palette= "Set1")
plt.title("Credit amount vs Age coloured by Sex and sized by Duration")
plt.show()

In [ ]:
sns.violinplot(data= df, x= "Saving accounts", y= "Credit amount", palette= "Pastel1")
plt.title("Credit Amount Distribution by Saving Accounts")
plt.show()

In [ ]:
df["Risk"].value_counts(normalize=True)*100

In [ ]:
plt.figure(figsize=(15,4))
for i, col in enumerate(["Age","Credit amount","Duration"]):
  plt.subplot(1,3, i+1)
  sns.boxplot(data=df,x="Risk", y= col, palette= "Pastel2")
  plt.title(f"{col} by Risk")

plt.tight_layout()
plt.show()

In [ ]:
df.groupby("Risk")[["Age","Credit amount","Duration"]].mean()

In [ ]:
categorical_cols

In [ ]:
plt.figure(figsize=(15,10))
for i, col in enumerate(categorical_cols):
  plt.subplot(3,3, i+1)
  sns.countplot(data=df, x= col, hue="Risk", palette= "Set1", order = df[col].value_counts().index)
  plt.title(f"{col} by Risk")
  plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
df.columns

In [ ]:
features= ["Age", "Sex","Job","Housing","Saving accounts","Checking account","Credit amount","Duration"]

In [ ]:
target= "Risk"

In [ ]:
df_model=df[features + [target]].copy()

In [ ]:
df_model.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder
import joblib

In [ ]:
cat_cols= df_model.select_dtypes(include="object").columns.drop("Risk")

In [ ]:
le_dict={}

In [ ]:
cat_cols

In [ ]:
for col in cat_cols:
  le= LabelEncoder()
  df_model[col]= le.fit_transform(df_model[col])
  le_dict[col]= le
  joblib.dump(le, f"{col}_encoder.pkl")

In [79]:
le_target= LabelEncoder()
df_model[target]= le_target.fit_transform(df_model[target])

In [ ]:
target

In [ ]:
df_model[target]

In [ ]:
df_model[target].value_counts()

In [ ]:
joblib.dump(le_target, "target_encoder.pkl")

In [ ]:
df_model.head()

In [ ]:
from sklearn.model_selection import train_test_split

In [82]:
X=df_model.drop(target, axis=1)

In [83]:
y=df_model[target]

In [ ]:
X

In [ ]:
y

In [84]:
X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.2, stratify=y, random_state=1)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV

In [ ]:
def train_model(model, param_grid, X_train, y_train, X_test, y_test):
  grid= GridSearchCV(model, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
  grid.fit(X_train, y_train)
  best_model= grid.best_estimator_
  y_pred= best_model.predict(X_test)
  acc= accuracy_score(y_test, y_pred)
  return best_model, acc, grid.best_params_

In [ ]:
dt = DecisionTreeClassifier(random_state=1, class_weight="balanced")
dt_param_grid = {
    'max_depth': [3,5,7,10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
best_dt, acc_dt, params_dt= train_model(dt, dt_param_grid, X_train, y_train, X_test, y_test)

In [ ]:
print("Decision Tree Accuracy", acc_dt)

In [ ]:
print("Best parameters", params_dt)

In [ ]:
rf= RandomForestClassifier(random_state=1, class_weight="balanced", n_jobs=-1)

In [ ]:
rf_param_grid={
    "n_estimators": [100,200],
    "max_depth": [5, 7, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1,2,4]
}

In [ ]:
best_rf, acc_rf, params_rf= train_model(rf, rf_param_grid, X_train, y_train, X_test, y_test)

In [ ]:
print("Random Forest Accuracy", acc_rf)

In [ ]:
print("Best params", params_rf)

In [ ]:
et= ExtraTreesClassifier(random_state=1, class_weight="balanced", n_jobs=-1)

In [ ]:
et_param_grid={
    "n_estimators": [100,200],
    "max_depth": [5, 7, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1,2,4]
}

In [73]:
best_et, acc_et, params_et= train_model(et, et_param_grid, X_train, y_train, X_test, y_test)

In [74]:
print("Extra trees accuracy", acc_et)

Extra trees accuracy 0.755


In [86]:
xgb= XGBClassifier(random_state=1, scale_pos_weight= (y_train ==1).sum(), use_label_encoder= False, eval_metric= "logloss")

In [87]:
xgb_param_grid={
    "n_estimators": [100,200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2],
    "subsample": [0.7, 1],
    "colsample_bytree": [0.7, 1]
}

In [88]:
best_xgb, acc_xgb, params_xgb= train_model(xgb, xgb_param_grid, X_train, y_train, X_test, y_test)

/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [20:41:31] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [89]:
print("XGB accuracy", acc_xgb)

XGB accuracy 0.725


In [92]:
print("Best params", params_xgb)

Best params {'colsample_bytree': 1, 'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.7}


In [91]:
best_et.predict(X_test)

array(['good', 'good', 'good', 'good', 'bad', 'good', 'good', 'good',
       'good', 'good', 'bad', 'good', 'good', 'good', 'bad', 'good',
       'good', 'good', 'good', 'bad', 'good', 'bad', 'good', 'good',
       'good', 'good', 'good', 'good', 'good', 'good', 'good', 'good',
       'good', 'good', 'good', 'good', 'bad', 'good', 'good', 'good',
       'good', 'bad', 'good', 'good', 'good', 'good', 'good', 'bad',
       'good', 'good', 'good', 'good', 'bad', 'good', 'bad', 'bad', 'bad',
       'good', 'good', 'good', 'good', 'good', 'good', 'good', 'good',
       'good', 'good', 'good', 'bad', 'good', 'bad', 'bad', 'good',
       'good', 'good', 'good', 'good', 'good', 'good', 'good', 'good',
       'bad', 'bad', 'good', 'bad', 'good', 'good', 'good', 'good',
       'good', 'good', 'good', 'good', 'good', 'good', 'good', 'good',
       'bad', 'good', 'bad', 'bad', 'good', 'bad', 'good', 'good', 'bad',
       'good', 'good', 'good', 'bad', 'good', 'good', 'bad', 'good',
       'good', 

In [93]:
joblib.dump(best_et, "extra_trees_credit_model.pkl")

['extra_trees_credit_model.pkl']